<a href="https://colab.research.google.com/github/eshan14git/football-qa-nlp/blob/eshan-dev/notebooks/03_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Feature Engineering for Football Question Answering Assistant

## Objective

This notebook transforms the preprocessed football datasets into machine-readable features that can be used by both Machine Learning and Deep Learning models.

The feature engineering process includes:

- Creating textual representations of football records
- Preparing text for NLP processing
- Generating TF-IDF features for the Random Forest model
- Creating tokenized and padded sequences for the LSTM model

These features will later be used for model training and evaluation.

In [1]:
# Data manipulation
import pandas as pd
import numpy as np

# Text processing
import re

# Feature Engineering
from sklearn.feature_extraction.text import TfidfVectorizer

# Deep Learning preprocessing
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Display options
pd.set_option("display.max_colwidth", None)

In [11]:
!git clone https://github.com/eshan14git/football-qa-nlp.git

fatal: destination path 'football-qa-nlp' already exists and is not an empty directory.


In [12]:
!ls

football-qa-nlp  sample_data


In [14]:
!ls football-qa-nlp/data

former_names.csv  goalscorers.csv  results.csv	shootouts.csv


In [15]:
!rm -rf football-qa-nlp
!git clone -b eshan-dev https://github.com/eshan14git/football-qa-nlp.git

Cloning into 'football-qa-nlp'...
remote: Enumerating objects: 137, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 137 (delta 18), reused 17 (delta 12), pack-reused 112 (from 4)
Receiving objects: 100% (137/137), 7.94 MiB | 14.86 MiB/s, done.
Resolving deltas: 100% (50/50), done.


In [16]:
!git clone -b eshan-dev https://github.com/eshan14git/football-qa-nlp.git
!ls football-qa-nlp/data

fatal: destination path 'football-qa-nlp' already exists and is not an empty directory.
former_names_clean.csv	goalscorers.csv    shootouts_clean.csv
former_names.csv	results_clean.csv  shootouts.csv
goalscorers_clean.csv	results.csv


In [17]:
# Load the cleaned datasets

results_clean = pd.read_csv("football-qa-nlp/data/results_clean.csv")
scorers_clean = pd.read_csv("football-qa-nlp/data/goalscorers_clean.csv")
shootouts_clean = pd.read_csv("football-qa-nlp/data/shootouts_clean.csv")
former_names_clean = pd.read_csv("football-qa-nlp/data/former_names_clean.csv")

In [18]:
print("Results:", results_clean.shape)
print("Goalscorers:", scorers_clean.shape)
print("Shootouts:", shootouts_clean.shape)
print("Former Names:", former_names_clean.shape)

Results: (49485, 9)
Goalscorers: (47855, 8)
Shootouts: (682, 5)
Former Names: (36, 4)


In [19]:
# Display column names for each cleaned dataset

print("Results Dataset Columns:")
print(results_clean.columns.tolist())

print("\nGoalscorers Dataset Columns:")
print(scorers_clean.columns.tolist())

print("\nShootouts Dataset Columns:")
print(shootouts_clean.columns.tolist())

print("\nFormer Names Dataset Columns:")
print(former_names_clean.columns.tolist())

Results Dataset Columns:
['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']

Goalscorers Dataset Columns:
['date', 'home_team', 'away_team', 'team', 'scorer', 'minute', 'own_goal', 'penalty']

Shootouts Dataset Columns:
['date', 'home_team', 'away_team', 'winner', 'first_shooter']

Former Names Dataset Columns:
['current', 'former', 'start_date', 'end_date']


In [21]:
# Create a separate DataFrame for match-result text documents
results_documents = results_clean.copy()

# Convert score columns to integers before inserting them into sentences
results_documents["home_score"] = results_documents["home_score"].astype(int)
results_documents["away_score"] = results_documents["away_score"].astype(int)

# Create one natural-language document for each match
results_documents["text"] = (
    "On " + results_documents["date"].astype(str)
    + ", " + results_documents["home_team"]
    + " played against " + results_documents["away_team"]
    + " in the " + results_documents["tournament"]
    + " in " + results_documents["city"]
    + ", " + results_documents["country"]
    + ". The final score was "
    + results_documents["home_team"] + " "
    + results_documents["home_score"].astype(str)
    + " and " + results_documents["away_team"] + " "
    + results_documents["away_score"].astype(str)
    + "."
)

# Add a source label to identify the dataset
results_documents["source"] = "match_result"

# Display sample generated documents
results_documents[["text", "source"]].head()

,text,source
0,"On 1872-11-30, Scotland played against England in the Friendly in Glasgow, Scotland. The final score was Scotland 0 and England 0.",match_result
1,"On 1873-03-08, England played against Scotland in the Friendly in London, England. The final score was England 4 and Scotland 2.",match_result
2,"On 1874-03-07, Scotland played against England in the Friendly in Glasgow, Scotland. The final score was Scotland 2 and England 1.",match_result
3,"On 1875-03-06, England played against Scotland in the Friendly in London, England. The final score was England 2 and Scotland 2.",match_result
4,"On 1876-03-04, Scotland played against England in the Friendly in Glasgow, Scotland. The final score was Scotland 3 and England 0.",match_result


In [22]:
# Create a separate DataFrame for goal-event text documents
scorers_documents = scorers_clean.copy()

# Create the opponent team dynamically
scorers_documents["opponent"] = np.where(
    scorers_documents["team"] == scorers_documents["home_team"],
    scorers_documents["away_team"],
    scorers_documents["home_team"]
)

# Convert the goal minute to text while preserving missing values
scorers_documents["minute"] = scorers_documents["minute"].fillna("Unknown").astype(str)

# Create one natural-language document for each goal event
scorers_documents["text"] = (
    "On " + scorers_documents["date"].astype(str)
    + ", " + scorers_documents["scorer"]
    + " scored for " + scorers_documents["team"]
    + " against " + scorers_documents["opponent"]
    + ". Goal minute: "
    + scorers_documents["minute"]
    + ". Penalty: "
    + scorers_documents["penalty"].astype(str)
    + ". Own goal: "
    + scorers_documents["own_goal"].astype(str)
    + "."
)

# Add a source label
scorers_documents["source"] = "goal_event"

# Display sample generated documents
scorers_documents[["text", "source"]].head()

,text,source
0,"On 1916-07-02, José Piendibene scored for Uruguay against Chile. Goal minute: 44. Penalty: False. Own goal: False.",goal_event
1,"On 1916-07-02, Isabelino Gradín scored for Uruguay against Chile. Goal minute: 55. Penalty: False. Own goal: False.",goal_event
2,"On 1916-07-02, Isabelino Gradín scored for Uruguay against Chile. Goal minute: 70. Penalty: False. Own goal: False.",goal_event
3,"On 1916-07-02, José Piendibene scored for Uruguay against Chile. Goal minute: 75. Penalty: False. Own goal: False.",goal_event
4,"On 1916-07-06, Alberto Ohaco scored for Argentina against Chile. Goal minute: 2. Penalty: False. Own goal: False.",goal_event


In [23]:
# Create a separate DataFrame for penalty-shootout text documents
shootouts_documents = shootouts_clean.copy()

# Create one natural-language document for each penalty shootout
shootouts_documents["text"] = (
    "On " + shootouts_documents["date"].astype(str)
    + ", " + shootouts_documents["home_team"]
    + " played against " + shootouts_documents["away_team"]
    + " in a penalty shootout. The shootout winner was "
    + shootouts_documents["winner"]
    + ". The first shooter was "
    + shootouts_documents["first_shooter"]
    + "."
)

# Add a source label
shootouts_documents["source"] = "shootout"

# Display sample generated documents
shootouts_documents[["text", "source"]].head()

,text,source
0,"On 1967-08-22, India played against Taiwan in a penalty shootout. The shootout winner was Taiwan. The first shooter was Unknown.",shootout
1,"On 1971-11-14, South Korea played against Vietnam Republic in a penalty shootout. The shootout winner was South Korea. The first shooter was Unknown.",shootout
2,"On 1972-05-07, South Korea played against Iraq in a penalty shootout. The shootout winner was Iraq. The first shooter was Unknown.",shootout
3,"On 1972-05-17, Thailand played against South Korea in a penalty shootout. The shootout winner was South Korea. The first shooter was Unknown.",shootout
4,"On 1972-05-19, Thailand played against Cambodia in a penalty shootout. The shootout winner was Thailand. The first shooter was Unknown.",shootout


In [24]:
# Combine all text documents into one corpus

football_corpus = pd.concat(
    [
        results_documents[["text", "source"]],
        scorers_documents[["text", "source"]],
        shootouts_documents[["text", "source"]]
    ],
    ignore_index=True
)

# Display corpus information
print("Total documents:", football_corpus.shape[0])

# Preview the combined corpus
football_corpus.head()

Total documents: 98022


,text,source
0,"On 1872-11-30, Scotland played against England in the Friendly in Glasgow, Scotland. The final score was Scotland 0 and England 0.",match_result
1,"On 1873-03-08, England played against Scotland in the Friendly in London, England. The final score was England 4 and Scotland 2.",match_result
2,"On 1874-03-07, Scotland played against England in the Friendly in Glasgow, Scotland. The final score was Scotland 2 and England 1.",match_result
3,"On 1875-03-06, England played against Scotland in the Friendly in London, England. The final score was England 2 and Scotland 2.",match_result
4,"On 1876-03-04, Scotland played against England in the Friendly in Glasgow, Scotland. The final score was Scotland 3 and England 0.",match_result


In [25]:
# Create a copy of the corpus for text cleaning
football_corpus_clean = football_corpus.copy()

# Convert all text to lowercase
football_corpus_clean["text"] = football_corpus_clean["text"].str.lower()

# Remove punctuation while preserving numbers, letters, spaces and hyphens
football_corpus_clean["text"] = football_corpus_clean["text"].str.replace(
    r"[^a-z0-9\s-]",
    "",
    regex=True
)

# Remove extra whitespace
football_corpus_clean["text"] = football_corpus_clean["text"].str.replace(
    r"\s+",
    " ",
    regex=True
).str.strip()

# Display sample cleaned documents
football_corpus_clean.head()

,text,source
0,on 1872-11-30 scotland played against england in the friendly in glasgow scotland the final score was scotland 0 and england 0,match_result
1,on 1873-03-08 england played against scotland in the friendly in london england the final score was england 4 and scotland 2,match_result
2,on 1874-03-07 scotland played against england in the friendly in glasgow scotland the final score was scotland 2 and england 1,match_result
3,on 1875-03-06 england played against scotland in the friendly in london england the final score was england 2 and scotland 2,match_result
4,on 1876-03-04 scotland played against england in the friendly in glasgow scotland the final score was scotland 3 and england 0,match_result


In [26]:
# Display basic information about the final corpus

print("Total documents:", len(football_corpus_clean))

print("\nDocument types:")
print(football_corpus_clean["source"].value_counts())

print("\nSample cleaned documents:")
display(football_corpus_clean.sample(5, random_state=42))

Total documents: 98022

Document types:
source
match_result    49485
goal_event      47855
shootout          682
Name: count, dtype: int64

Sample cleaned documents:


,text,source
23781,on 1999-08-03 myanmar played against vietnam in the southeast asian games in bandar seri begawan brunei the final score was myanmar 0 and vietnam 2,match_result
84519,on 2015-10-09 raheem sterling scored for england against estonia goal minute 85 penalty false own goal false,goal_event
93441,on 2023-11-17 federico chiesa scored for italy against north macedonia goal minute 41 penalty false own goal false,goal_event
97389,on 1982-02-22 ivory coast played against burkina faso in a penalty shootout the shootout winner was burkina faso the first shooter was unknown,shootout
38388,on 2014-11-26 malaysia played against thailand in the aff championship in kallang singapore the final score was malaysia 2 and thailand 3,match_result


In [27]:
# Create the TF-IDF vectorizer

tfidf_vectorizer = TfidfVectorizer(
    lowercase=False,
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2
)

# Learn the vocabulary and transform the corpus into TF-IDF features
tfidf_features = tfidf_vectorizer.fit_transform(football_corpus_clean["text"])

# Display the feature matrix shape
print("TF-IDF feature matrix shape:", tfidf_features.shape)

TF-IDF feature matrix shape: (98022, 5000)


In [28]:
# Display the number of extracted features
print("Number of TF-IDF features:", len(tfidf_vectorizer.get_feature_names_out()))

# Display the first 20 features
print("\nFirst 20 TF-IDF features:")
print(tfidf_vectorizer.get_feature_names_out()[:20])

Number of TF-IDF features: 5000

First 20 TF-IDF features:
['01' '01 04' '01 05' '01 06' '01 07' '01 08' '01 09' '01 10' '01 11'
 '01 12' '01 13' '01 14' '01 15' '01 16' '01 17' '01 18' '01 19' '01 20'
 '01 21' '01 22']


In [29]:
# Store all extracted feature names
feature_names = tfidf_vectorizer.get_feature_names_out()

# Display selected football-related features
football_terms = [
    feature for feature in feature_names
    if any(
        keyword in feature
        for keyword in [
            "world cup",
            "goal",
            "penalty",
            "shootout",
            "england",
            "argentina",
            "friendly"
        ]
    )
]

print("Sample football-related TF-IDF features:")
print(football_terms[:50])

Sample football-related TF-IDF features:
['10 penalty', '11 penalty', '12 penalty', '13 penalty', '14 penalty', '15 penalty', '16 penalty', '17 penalty', '18 penalty', '19 penalty', '20 penalty', '21 penalty', '22 penalty', '23 penalty', '24 penalty', '25 penalty', '26 penalty', '27 penalty', '28 penalty', '29 penalty', '30 penalty', '31 penalty', '32 penalty', '33 penalty', '34 penalty', '35 penalty', '36 penalty', '37 penalty', '38 penalty', '39 penalty', '40 penalty', '41 penalty', '42 penalty', '43 penalty', '44 penalty', '45 penalty', '46 penalty', '47 penalty', '48 penalty', '49 penalty', '50 penalty', '51 penalty', '52 penalty', '53 penalty', '54 penalty', '55 penalty', '56 penalty', '57 penalty', '58 penalty', '59 penalty']


In [30]:
# Store all extracted feature names
feature_names = tfidf_vectorizer.get_feature_names_out()

# Show exact football-related words and phrases
important_terms = [
    "goal",
    "goal minute",
    "own goal",
    "penalty",
    "penalty shootout",
    "shootout winner",
    "world cup",
    "fifa world",
    "england",
    "argentina",
    "friendly"
]

found_terms = [
    term for term in important_terms
    if term in feature_names
]

print("Important football-related TF-IDF features:")
print(found_terms)

Important football-related TF-IDF features:
['goal', 'goal minute', 'own goal', 'penalty', 'penalty shootout', 'shootout winner', 'world cup', 'fifa world', 'england', 'argentina', 'friendly']


In [31]:
import os
import joblib
from scipy import sparse

In [32]:
# Create a folder to store engineered features
features_path = "football-qa-nlp/models/features"

os.makedirs(features_path, exist_ok=True)

# Save the cleaned football corpus
football_corpus_clean.to_csv(
    f"{features_path}/football_corpus_clean.csv",
    index=False
)

# Save the fitted TF-IDF vectorizer
joblib.dump(
    tfidf_vectorizer,
    f"{features_path}/tfidf_vectorizer.pkl"
)

# Save the sparse TF-IDF feature matrix
sparse.save_npz(
    f"{features_path}/tfidf_features.npz",
    tfidf_features
)

print("✅ TF-IDF artifacts saved successfully.")

✅ TF-IDF artifacts saved successfully.


In [33]:
# Create the tokenizer for LSTM input preparation
lstm_tokenizer = Tokenizer(
    num_words=10000,
    oov_token="<OOV>"
)

# Learn the vocabulary from the cleaned football corpus
lstm_tokenizer.fit_on_texts(football_corpus_clean["text"])

# Display tokenizer information
print("Total words found:", len(lstm_tokenizer.word_index))
print("Vocabulary limit used:", 10000)
print("OOV token index:", lstm_tokenizer.word_index.get("<OOV>"))

Total words found: 19794
Vocabulary limit used: 10000
OOV token index: 1


In [34]:
# Display the first 20 learned word-to-index mappings
first_words = list(lstm_tokenizer.word_index.items())[:20]

print("First 20 tokenizer word mappings:")

for word, index in first_words:
    print(f"{word}: {index}")

First 20 tokenizer word mappings:
<OOV>: 1
the: 2
in: 3
on: 4
against: 5
goal: 6
false: 7
and: 8
was: 9
played: 10
final: 11
score: 12
penalty: 13
scored: 14
for: 15
minute: 16
own: 17
1: 18
0: 19
06: 20


In [35]:
# Convert every cleaned document into a sequence of integer word IDs
lstm_sequences = lstm_tokenizer.texts_to_sequences(
    football_corpus_clean["text"]
)

# Display one original document and its integer sequence
print("Original document:")
print(football_corpus_clean["text"].iloc[0])

print("\nInteger sequence:")
print(lstm_sequences[0])

Original document:
on 1872-11-30 scotland played against england in the friendly in glasgow scotland the final score was scotland 0 and england 0

Integer sequence:
[4, 1, 25, 100, 90, 10, 5, 56, 3, 2, 24, 3, 475, 90, 2, 11, 12, 9, 90, 19, 8, 56, 19]


In [36]:
# Calculate the number of tokens in every document
sequence_lengths = np.array(
    [len(sequence) for sequence in lstm_sequences]
)

# Display sequence-length statistics
print("Shortest sequence:", sequence_lengths.min())
print("Longest sequence:", sequence_lengths.max())
print("Average sequence length:", sequence_lengths.mean())
print("Median sequence length:", np.median(sequence_lengths))
print("95th percentile:", np.percentile(sequence_lengths, 95))
print("99th percentile:", np.percentile(sequence_lengths, 99))

Shortest sequence: 18
Longest sequence: 44
Average sequence length: 22.70575993144396
Median sequence length: 23.0
95th percentile: 29.0
99th percentile: 33.0


In [37]:
# Define the maximum sequence length
max_sequence_length = 33

# Pad all sequences to the same length
lstm_padded_sequences = pad_sequences(
    lstm_sequences,
    maxlen=max_sequence_length,
    padding="post",
    truncating="post"
)

# Display the padded sequence shape
print("Padded sequence shape:", lstm_padded_sequences.shape)

# Display the first padded sequence
print("\nFirst padded sequence:")
print(lstm_padded_sequences[0])


Padded sequence shape: (98022, 33)

First padded sequence:
[  4   1  25 100  90  10   5  56   3   2  24   3 475  90   2  11  12   9
  90  19   8  56  19   0   0   0   0   0   0   0   0   0   0]


In [38]:
# Save the fitted LSTM tokenizer
joblib.dump(
    lstm_tokenizer,
    f"{features_path}/lstm_tokenizer.pkl"
)

# Save the padded LSTM sequences
np.save(
    f"{features_path}/lstm_padded_sequences.npy",
    lstm_padded_sequences
)

# Save the sequence-length setting
joblib.dump(
    max_sequence_length,
    f"{features_path}/max_sequence_length.pkl"
)

print("LSTM feature artifacts saved successfully.")

LSTM feature artifacts saved successfully.


In [39]:
# Display all saved feature-engineering files
print("Saved feature artifacts:\n")

for filename in sorted(os.listdir(features_path)):
    print(filename)

Saved feature artifacts:

football_corpus_clean.csv
lstm_padded_sequences.npy
lstm_tokenizer.pkl
max_sequence_length.pkl
tfidf_features.npz
tfidf_vectorizer.pkl
